# N-Body Gravitational Simulation

Numerically simulate multiple bodies interacting through mutual gravitational attraction in 2D.

Uses velocity-Verlet integration for numerical stability.

**Units:**
- Distance: AU (Astronomical Units)
- Time: years
- Mass: solar masses
- With these units, $G = 4\pi^2$

**Example:** Simulate Earth, Mars, Jupiter orbiting the Sun!

In [1]:
import numpy as np
import matplotlib.pyplot as plt
from ipywidgets import FloatSlider, IntSlider, Checkbox, VBox, HBox, Output
from IPython.display import display, Markdown

print('Environment ready for N-body simulation.')

Environment ready for N-body simulation.


In [2]:
G = 4 * np.pi**2  # AU^3 / (solar_mass * year^2)
EPS = 1e-6  # softening parameter


def accelerations(positions, masses):
    n = len(masses)
    acc = np.zeros_like(positions)
    for i in range(n):
        for j in range(n):
            if i == j:
                continue
            r = positions[j] - positions[i]
            dist2 = np.dot(r, r) + EPS
            dist = np.sqrt(dist2)
            acc[i] += G * masses[j] * r / (dist2 * dist)
    return acc


def simulate_nbody(masses, pos0, vel0, dt, steps):
    n = len(masses)
    pos = np.zeros((steps + 1, n, 2))
    vel = np.zeros((steps + 1, n, 2))

    pos[0] = pos0
    vel[0] = vel0

    a = accelerations(pos[0], masses)

    for s in range(steps):
        pos[s + 1] = pos[s] + vel[s] * dt + 0.5 * a * dt**2
        a_new = accelerations(pos[s + 1], masses)
        vel[s + 1] = vel[s] + 0.5 * (a + a_new) * dt
        a = a_new

    return pos, vel


# Preset solar system configuration
base_names = ['Sun', 'Earth', 'Mars', 'Jupiter']
base_masses = np.array([1.0, 3.003e-6, 3.227e-7, 9.545e-4])
base_pos = np.array([
    [0.0, 0.0],
    [1.0, 0.0],
    [1.524, 0.0],
    [5.203, 0.0],
])
base_vel = np.array([
    [0.0, 0.0],
    [0.0, 2 * np.pi],
    [0.0, 2 * np.pi / np.sqrt(1.524)],
    [0.0, 2 * np.pi / np.sqrt(5.203)],
])

# Controls
years_slider = FloatSlider(value=12.0, min=1.0, max=100.0, step=1.0, description='Years')
dt_slider = FloatSlider(value=0.002, min=0.0005, max=0.02, step=0.0005, readout_format='.4f', description='dt')
planet_count_slider = IntSlider(value=4, min=2, max=4, step=1, description='Bodies')
equal_axes = Checkbox(value=True, description='Equal axis scale')
show_sun_trail = Checkbox(value=False, description='Show Sun trail')

ui = VBox([
    HBox([years_slider, dt_slider]),
    HBox([planet_count_slider, equal_axes, show_sun_trail])
])
display(ui)

out = Output()


def update_plot(*_):
    with out:
        out.clear_output(wait=True)

        n = planet_count_slider.value
        masses = base_masses[:n]
        names = base_names[:n]
        pos0 = base_pos[:n].copy()
        vel0 = base_vel[:n].copy()

        # Shift total momentum to approximately zero
        total_momentum = np.sum(masses[:, None] * vel0, axis=0)
        vel0[0] -= total_momentum / masses[0]

        years = years_slider.value
        dt = dt_slider.value
        steps = max(1, int(years / dt))

        pos, vel = simulate_nbody(masses, pos0, vel0, dt, steps)

        fig, ax = plt.subplots(figsize=(9, 9))
        colors = ['gold', 'tab:blue', 'tab:red', 'tab:orange']

        for i in range(n):
            if i == 0 and not show_sun_trail.value:
                pass
            else:
                ax.plot(pos[:, i, 0], pos[:, i, 1], lw=1.2, color=colors[i], label=f'{names[i]} path')
            ax.scatter(pos[-1, i, 0], pos[-1, i, 1], s=70 if i == 0 else 35, color=colors[i])

        ax.set_title(f'N-body Gravitational Simulation ({n} bodies, {years:.1f} years)')
        ax.set_xlabel('x (AU)')
        ax.set_ylabel('y (AU)')
        ax.grid(True, alpha=0.3)

        if equal_axes.value:
            ax.set_aspect('equal', adjustable='box')

        ax.legend(loc='upper right')
        plt.show()

        if n > 1:
            r_earth = np.sqrt(np.sum(pos[:, 1, :]**2, axis=1))
            display(Markdown(f'**Simulation steps:** {steps}'))
            display(Markdown(f'**Time step dt:** {dt:.4f} years'))
            display(Markdown(f'**Earth orbital radius range:** [{r_earth.min():.3f}, {r_earth.max():.3f}] AU'))


for w in [years_slider, dt_slider, planet_count_slider, equal_axes, show_sun_trail]:
    w.observe(update_plot, names='value')

update_plot()
display(out)

Output()